# Social Media Sentiment Analysis
## Week 2: Model Training & Prediction

---

- Convert cleaned text into numerical features (TF-IDF)
- Train multiple ML classification models
- Evaluate and compare model performance
- Select the best model
- Test live predictions on new text

---

**Models trained:** Logistic Regression, Naive Bayes, Random Forest, Linear SVM

In [ ]:
!pip install scikit-learn nltk pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
import pickle


In [ ]:
try:
    df = pd.read_csv('week1_processed_data.csv')
    df = df.dropna(subset=['cleaned_text', 'sentiment'])
    df['cleaned_text'] = df['cleaned_text'].astype(str)
    print(f'Loaded week1_processed_data.csv — {len(df):,} records')
except FileNotFoundError:
    print('week1_processed_data.csv not found.')
    print('Please upload the Week 1 output CSV first.')
    print('Generating fallback sample data...')
    import nltk
    from nltk.corpus import stopwords
    nltk.download('stopwords', quiet=True)

    sentiments = (['Positive']*10 + ['Negative']*10 + ['Neutral']*10) * 4
    df = pd.DataFrame({'cleaned_text': texts[:len(sentiments)], 'sentiment': sentiments})
    print(f'Fallback sample created: {len(df)} records')

print(df['sentiment'].value_counts())
df.head()

In [ ]:
label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['label'] = df['sentiment'].map(label_map)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train):,}')
print(f'Testing samples  : {len(X_test):,}')
print()

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'TF-IDF feature matrix shape: {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,} words')

mean_scores = np.asarray(X_train_tfidf.mean(axis=0)).flatten()
top_idx = mean_scores.argsort()[-10:][::-1]
vocab_inv = {v: k for k, v in tfidf.vocabulary_.items()}
for i in top_idx:
    print(f'  {vocab_inv[i]:<25} score: {mean_scores[i]:.4f}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.5),
    'Linear SVM':          LinearSVC(max_iter=2000, C=1.0, random_state=42)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'predictions': y_pred, 'accuracy': acc}
    print(f'  Accuracy: {acc*100:.2f}%')


In [ ]:
label_names = ['Negative', 'Neutral', 'Positive']

for name, res in results.items():
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(classification_report(y_test, res['predictions'],
                                target_names=label_names, zero_division=0))

In [ ]:
names = list(results.keys())
accuracies = [results[n]['accuracy'] * 100 for n in names]
best_name = names[np.argmax(accuracies)]

bar_colors = ['#e74c3c' if n != best_name else '#2ecc71' for n in names]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, accuracies, color=bar_colors, edgecolor='black', width=0.5)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=11)
plt.ylim(0, 110)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=10)
plt.axhline(y=max(accuracies), color='green', linestyle='--', alpha=0.4, label=f'Best: {max(accuracies):.2f}%')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('week2_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best model: {best_name} ({max(accuracies):.2f}%)')

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('week2_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_model = results[best_name]['model']
print(f'Best model selected: {best_name}')
print(f'Test accuracy      : {results[best_name]["accuracy"]*100:.2f}%')

with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)




In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
for pkg in ['stopwords','wordnet','omw-1.4']:
    nltk.download(pkg, quiet=True)

stop_words   = set(stopwords.words('english'))
lemmatizer   = WordNetLemmatizer()
label_decode = {2: 'POSITIVE', 1: 'NEUTRAL', 0: 'NEGATIVE'}
emoji_map    = {'POSITIVE': '', 'NEUTRAL': '', 'NEGATIVE': ''}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = text.encode('ascii','ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [lemmatizer.lemmatize(w) for w in text.split()
              if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

def predict_sentiment(texts):
    cleaned  = [preprocess(t) for t in texts]
    features = tfidf.transform(cleaned)
    labels   = best_model.predict(features)
    print(f'\n{"="*60}')
    print(f'  SENTIMENT PREDICTIONS — {best_name}')
    print(f'{"="*60}')
    for text, label in zip(texts, labels):
        sentiment = label_decode[label]
        print(f'  [{sentiment}]  {text[:65]}')
    print()

test_texts = [
    'I absolutely love this new phone, best smartphone I have ever used!',
    'Terrible customer service, they never replied to my complaint.',
    'The app was updated today with some minor changes.',
    'This product exceeded all my expectations, highly recommend!',
    'Completely useless, stopped working after one day. Very angry.',
    'The package arrived on time. Nothing special to report.',
    'Amazing deal! Got 50% off and the quality is outstanding.',
    'Worst experience ever at this restaurant. Food was awful.',
    'Just bought tickets for the event next month.',
    'Really happy with my purchase, will definitely buy again!'
]

predict_sentiment(test_texts)

In [ ]:
best_preds = results[best_name]['predictions']
pred_labels = [label_decode[p] for p in best_preds]
actual_labels = [label_decode[a] for a in y_test]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Prediction Results — {best_name}', fontsize=14, fontweight='bold')

colors = {'POSITIVE':'#2ecc71','NEGATIVE':'#e74c3c','NEUTRAL':'#95a5a6'}

for ax, data, title in zip(axes,
                            [actual_labels, pred_labels],
                            ['Actual Labels', 'Predicted Labels']):
    counts = pd.Series(data).value_counts()
    ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
           colors=[colors[l] for l in counts.index],
           startangle=140, explode=[0.04]*len(counts))
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('week2_prediction_distribution.png', dpi=150, bbox_inches='tight')
plt.show()